In [1]:
import os
from matplotlib import pyplot as plt
import numpy as np
import pynwb
import h5py
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA

prefix = f'{os.getcwd()}/../data/{os.path.basename(os.getcwd())}/'
regen = False

# read behavior data

In [15]:
if regen:
	path = prefix + 'exported_data_gcamp_freely_moving/'
	os.makedirs(prefix + 'hdf5-behavior', exist_ok=True)
	filenames = os.listdir(path)
	for i, filename in enumerate(filenames[6:]):
		print(f'processing {i}: {filename}')
		dirpath = prefix + 'hdf5-behavior/' + os.path.splitext(filename)[0]
		os.makedirs(dirpath, exist_ok=True)
		nwbfile = pynwb.read_nwb(path + filename)
		if 'Behavior' not in nwbfile.processing.keys():
			print(f'No behavior data in {filename}')
			continue
		behavioral_data = nwbfile.processing['Behavior'].data_interfaces
		for key in behavioral_data:
			print(f'\t\t{key}')
			hdf5_data = behavioral_data[key].time_series[key].data
			with h5py.File(f'{dirpath}/{key}.h5', 'w') as h5file:
				h5file.create_dataset(key, data=hdf5_data)

else:
	behavior_data = list()

	path = prefix + 'hdf5-behavior/'
	foldernames = os.listdir(path)
	for foldername in foldernames:
		behavior_dict = dict()
		for filename in os.listdir(path + foldername):
			with h5py.File(path + foldername + '/' + filename, 'r') as h5file:
				filename_base = os.path.splitext(filename)[0]
				dataset = h5file[filename_base][:]
				behavior_dict[filename_base] = dataset
		behavior_data.append(behavior_dict)

# read calcium data

In [3]:
if regen:
	path = prefix + 'exported_data_gcamp_freely_moving/'

	calcium_data = list()
	os.makedirs(prefix + 'ndarray-calcium', exist_ok=True)
	filenames = os.listdir(path)
	for i, filename in enumerate(filenames):
		nwbfile = pynwb.read_nwb(path + filename)
		calcium_data = nwbfile.processing['CalciumActivity'].data_interfaces
		signal = np.array(calcium_data['SignalDFoF'].roi_response_series['SignalCalciumImResponseSeries'].data)
		calcium_data.append(signal)
		np.save(prefix + f'ndarray-calcium/{filename}.npy', signal)

else:
	calcium_data = list()

	path = prefix + 'ndarray-calcium/'
	for filename in os.listdir(path):
		calcium_npy = np.load(prefix + f'ndarray-calcium/{filename}')
		calcium_data.append(calcium_npy)

# stuff

In [22]:
sample_behavior = behavior_data[0]
sample_calcium = calcium_data[0]


for i, test in enumerate(behavior_data):
	if len(test) == 0: continue
	discr = test['discrete_states']
	for j, row in enumerate(discr):
		broke = False
		for elem in row:
			if not (np.isnan(elem) or elem == 0):
				broke = True
				break
		if broke:
			print(row)
			print(f'{i}, {j}')

[0. 0. 0. 0. 1. 0.]
10, 0
[0. 0. 0. 0. 1. 0.]
14, 0
[0. 0. 0. 0. 1. 0.]
18, 0
[0. 0. 0. 0. 1. 0.]
21, 0
[0. 0. 0. 0. 1. 0.]
29, 0
[0. 0. 0. 1. 0. 0.]
32, 0
